<a href="https://colab.research.google.com/github/2303a52506/HPC/blob/main/WEEK_3_HPC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from numba import njit, prange
import time


@njit
def matmul_serial(A, B):
    n, m = A.shape
    m2, p = B.shape
    C = np.zeros((n, p))

    for i in range(n):
        for j in range(p):
            for k in range(m):
                C[i, j] += A[i, k] * B[k, j]
    return C



@njit(parallel=True)
def matmul_parallel_outer(A, B):
    n, m = A.shape
    p = B.shape[1]
    C = np.zeros((n, p))

    for i in prange(n):          # Parallelized outer loop
        for j in range(p):
            for k in range(m):
                C[i, j] += A[i, k] * B[k, j]
    return C

@njit(parallel=True)
def matmul_parallel_collapsed(A, B):
    n, m = A.shape
    p = B.shape[1]
    C = np.zeros((n, p))

    for idx in prange(n * p):    # Collapse i,j into one loop
        i = idx // p
        j = idx % p
        for k in range(m):
            C[i, j] += A[i, k] * B[k, j]
    return C



N = 400
A = np.random.rand(N, N)
B = np.random.rand(N, N)


matmul_serial(A, B)
matmul_parallel_outer(A, B)
matmul_parallel_collapsed(A, B)

start = time.time()
matmul_serial(A, B)
print("Serial Time:", time.time() - start)

start = time.time()
matmul_parallel_outer(A, B)
print("Parallel Outer Loop Time:", time.time() - start)

start = time.time()
matmul_parallel_collapsed(A, B)
print("Parallel Collapsed Loop Time:", time.time() - start)

Serial Time: 0.14424371719360352
Parallel Outer Loop Time: 0.12096071243286133
Parallel Collapsed Loop Time: 0.13643479347229004
